## Dataset Preperation
<hr>
Dataset containing Last.fm track tags and song similarity from: http://millionsongdataset.com/lastfm/
<ul>
<li>943,347 matched tracks MSD <-> Last.fm</li>
<li>505,216 tracks with at least one tag</li>
<li>584,897 tracks with at least one similar track</li>
<li>522,366 unique tags</li>
<li>8,598,630 (track - tag) pairs</li>
<li>56,506,688 (track - similar track) pairs</li>
</ul>

SQLite database containing song lyrics in bag-of-words format from: http://millionsongdataset.com/musixmatch/
<ul>
<li>210,519 training bag-of-words</li>
<li>27,143 testing ones</li>
</ul>

SQLite database containing most metadata about each track from: http://millionsongdataset.com/pages/getting-dataset/

In [ ]:
pip install pylast; pandas; numpy; scipy; scikit-learn

In [24]:
import numpy as np
import pandas as pd
import sqlite3
import zipfile
import json
import pickle
from pathlib import Path
from scipy.sparse import save_npz, csr_matrix, vstack

<b>Extracting from the zip files</b>

In [6]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXTRACTED_DIR = RAW_DIR / "extracted"

for folder in (RAW_DIR, PROCESSED_DIR, EXTRACTED_DIR):
    folder.mkdir(parents=True, exist_ok=True)

In [7]:
print("Files in data/raw:")
for path in sorted(RAW_DIR.iterdir()):
    if path.is_file():
        print(f"  {path.name} ({path.stat().st_size / (1024 ** 2):.2f} MB)")
    else:
        print(f"  {path.name}/")

zip_files = [
    RAW_DIR / "lastfm_train.zip",
    RAW_DIR / "lastfm_test.zip",
]

db_files = list(RAW_DIR.glob("*.db"))

assert all(path.is_file() for path in zip_files), (
    "One or both Last.fm ZIP files are missing."
)
assert db_files, "No .db file found in data/raw."

DB_PATH = db_files[0]

print("\nDatabase selected:", DB_PATH.name)

Files in data/raw:
  extracted/
  lastfm_test.zip (131.96 MB)
  lastfm_train.zip (1028.37 MB)
  mxm_dataset.db (2328.45 MB)
  track_metadata.db (711.61 MB)

Database selected: mxm_dataset.db


<b>Extracting and exploring our JSON files</b>

In [8]:
for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(EXTRACTED_DIR)

    print("Extracted:", zip_path.name, "->", EXTRACTED_DIR)

Extracted: lastfm_train.zip -> c:\Users\Mina\Documents\CSU_MSDS\Found of AI\Music-Recommendation-ML-Project\data\raw\extracted
Extracted: lastfm_test.zip -> c:\Users\Mina\Documents\CSU_MSDS\Found of AI\Music-Recommendation-ML-Project\data\raw\extracted


In [9]:
lastfm_train_dir = EXTRACTED_DIR / "lastfm_train"
lastfm_test_dir = EXTRACTED_DIR / "lastfm_test"

train_json_files = list(lastfm_train_dir.rglob("*.json"))
test_json_files = list(lastfm_test_dir.rglob("*.json"))

print("Training JSON files:", len(train_json_files))
print("Testing JSON files:", len(test_json_files))

Training JSON files: 1678244
Testing JSON files: 208424


In [10]:
json_files = list(EXTRACTED_DIR.rglob("*.json"))

tag_examples = []

for file_path in json_files[:10]:
    with open(file_path, "r", encoding="utf-8") as f:
        record = json.load(f)

    tags = record.get("tags")

    tag_examples.append({
        "track_id": record.get("track_id", file_path.stem),
        "title": record.get("title"),
        "artist": record.get("artist"),
        "tags_type": type(tags).__name__,
        "tags_count": len(tags) if isinstance(tags, (list, dict)) else None,
        "tags_sample": str(tags)[:300],
    })

display(pd.DataFrame(tag_examples))

,track_id,title,artist,tags_type,tags_count,tags_sample
0,TRAAAEA128F935A30D,I'll Slap Your Face (Entertainment USA Theme),Jonathan King,list,0,[]
1,TRAAAQN128F9353BA0,Double Wide,Supersuckers,list,0,[]
2,TRAAAQO12903CD8E1C,Take Time,Rufus,list,0,[]
3,TRAAAZU128F4226F7A,Stop Laughing,Moose,list,0,[]
4,TRAABWZ12903CCDE85,Priority,Julien Lourau,list,0,[]
5,TRAABYL128F9340AAD,Stay (Live Version),Rufus,list,0,[]
6,TRAACAS128F4216F17,Dr Rudeboy,MRK1,list,6,"[['dubstep', '100'], ['UK', '25'], ['Grime', '..."
7,TRAACBA12903CFA373,I Don't Need A Cure For This,Radio Slave,list,3,"[['tech house', '100'], ['elektronischer-b0lze..."
8,TRAACUI128EF367904,The Shepherd's Dead,Payola,list,0,[]
9,TRAADAV128F4271E53,Qué Bueno Baila Usted,Ibrahim Ferrer,list,7,"[['latin', '100'], ['world', '100'], ['latin j..."


<b>Exploring data from the database file</b>

In [11]:
with sqlite3.connect(DB_PATH) as conn:
    tables = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
    """).fetchall()

print("Tables in database:")
for (table_name,) in tables:
    print(" ", table_name)

Tables in database:
  lyrics
  words


In [12]:
with sqlite3.connect(DB_PATH) as conn:
    for table_name in ["words", "lyrics"]:
        print(f"\n{'=' * 40}")
        print(f"TABLE: {table_name}")
        print("=" * 40)

        columns = conn.execute(
            f"PRAGMA table_info({table_name})"
        ).fetchall()

        print("\nColumns:")
        for column in columns:
            print(column)

        print("\nSample rows:")
        for row in conn.execute(
            f"SELECT * FROM {table_name} LIMIT 5"
        ):
            print(row)

        count = conn.execute(
            f"SELECT COUNT(*) FROM {table_name}"
        ).fetchone()[0]

        print("\nRow count:", count)


TABLE: words

Columns:
(0, 'word', 'TEXT', 0, None, 1)

Sample rows:
('i',)
('the',)
('you',)
('to',)
('and',)

Row count: 5000

TABLE: lyrics

Columns:
(0, 'track_id', '', 0, None, 0)
(1, 'mxm_tid', 'INT', 0, None, 0)
(2, 'word', 'TEXT', 0, None, 0)
(3, 'count', 'INT', 0, None, 0)
(4, 'is_test', 'INT', 0, None, 0)

Sample rows:
('TRAAAAV128F421A322', 4623710, 'i', 6, 0)
('TRAAAAV128F421A322', 4623710, 'the', 4, 0)
('TRAAAAV128F421A322', 4623710, 'you', 2, 0)
('TRAAAAV128F421A322', 4623710, 'to', 2, 0)
('TRAAAAV128F421A322', 4623710, 'and', 5, 0)

Row count: 19045332


<b>Loading and saving the vocabulary</b>

In [13]:
with sqlite3.connect(DB_PATH) as conn:
    vocab = [
        row[0]
        for row in conn.execute(
            "SELECT word FROM words ORDER BY ROWID"
        ).fetchall()
    ]

print("Vocabulary size:", len(vocab))
print("First 10 words:", vocab[:10])
print("Last 10 words:", vocab[-10:])

word_to_col = {word: i for i, word in enumerate(vocab)}

assert len(vocab) == 5000, "Expected 5,000 vocabulary words."
assert len(word_to_col) == 5000, "Vocabulary contains duplicate words."

with sqlite3.connect(DB_PATH) as conn:
    lyrics_df = pd.read_sql_query(
        """
        SELECT track_id, word, count, is_test
        FROM lyrics
        """,
        conn,
    )

lyrics_df["col"] = lyrics_df["word"].map(word_to_col)

assert lyrics_df["col"].notna().all(), (
    "Some lyric words are missing from the vocabulary."
)

lyrics_df["col"] = lyrics_df["col"].astype(np.int32)

print("\nLyric rows loaded:", len(lyrics_df))
print("Training lyric rows:", int((lyrics_df["is_test"] == 0).sum()))
print("Testing lyric rows:", int((lyrics_df["is_test"] == 1).sum()))

Vocabulary size: 5000
First 10 words: ['i', 'the', 'you', 'to', 'and', 'a', 'me', 'it', 'not', 'in']
Last 10 words: ['writer', 'motivo', 'bake', 'insist', 'wel', 'santo', 'pe', 'gee', 'colleg', 'kad']

Lyric rows loaded: 19045332
Training lyric rows: 16845943
Testing lyric rows: 2199389


In [14]:
def build_split_matrix(df, is_test, word_count):
    split_df = df[df["is_test"] == is_test]

    track_ids = sorted(split_df["track_id"].unique())
    track_to_row = {track_id: i for i, track_id in enumerate(track_ids)}

    rows = split_df["track_id"].map(track_to_row).to_numpy(
        dtype=np.int32
    )
    cols = split_df["col"].to_numpy(dtype=np.int32)
    counts = split_df["count"].to_numpy(dtype=np.int32)

    matrix = csr_matrix(
        (counts, (rows, cols)),
        shape=(len(track_ids), word_count),
        dtype=np.int32,
    )

    return track_ids, matrix

train_ids, X_train = build_split_matrix(
    lyrics_df, is_test=0, word_count=len(vocab)
)

test_ids, X_test = build_split_matrix(
    lyrics_df, is_test=1, word_count=len(vocab)
)

print("Training matrix:", X_train.shape)
print("Testing matrix:", X_test.shape)
print("Vocabulary size:", len(vocab))
print("Training nonzero entries:", X_train.nnz)
print("Testing nonzero entries:", X_test.nnz)

Training matrix: (210519, 5000)
Testing matrix: (27143, 5000)
Vocabulary size: 5000
Training nonzero entries: 16845943
Testing nonzero entries: 2199389


In [15]:
save_npz(PROCESSED_DIR / "lyrics_train.npz", X_train)
save_npz(PROCESSED_DIR / "lyrics_test.npz", X_test)

with open(PROCESSED_DIR / "lyrics_metadata.pkl", "wb") as f:
    pickle.dump(
        {
            "train_ids": train_ids,
            "test_ids": test_ids,
            "vocab": vocab,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print("Saved processed lyric data to:", PROCESSED_DIR.resolve())
for path in sorted(PROCESSED_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size / (1024 ** 2):.2f} MB")

Saved processed lyric data to: C:\Users\Mina\Documents\CSU_MSDS\Found of AI\Music-Recommendation-ML-Project\data\processed
lyrics_metadata.pkl: 4.80 MB
lyrics_test.npz: 4.15 MB
lyrics_train.npz: 31.78 MB
track_metadata_lookup.csv: 18.42 MB


In [16]:
del lyrics_df

<b>Exploring the track metadata file</b>

In [17]:
TRACK_METADATA_PATH = RAW_DIR / "track_metadata.db"

with sqlite3.connect(TRACK_METADATA_PATH) as conn:
    tables = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
    """).fetchall()

    print("Tables:", tables)

    columns = conn.execute(
        "PRAGMA table_info(songs)"
    ).fetchall()

    print("\nSongs table columns:")
    for column in columns:
        print(column)

    print("\First five rows:")
    for row in conn.execute("""
        SELECT track_id, title, artist_name
        FROM songs
        LIMIT 5
    """):
        print(row)

Tables: [('songs',)]

Songs table columns:
(0, 'track_id', 'TEXT', 0, None, 1)
(1, 'title', 'TEXT', 0, None, 0)
(2, 'song_id', 'TEXT', 0, None, 0)
(3, 'release', 'TEXT', 0, None, 0)
(4, 'artist_id', 'TEXT', 0, None, 0)
(5, 'artist_mbid', 'TEXT', 0, None, 0)
(6, 'artist_name', 'TEXT', 0, None, 0)
(7, 'duration', 'REAL', 0, None, 0)
(8, 'artist_familiarity', 'REAL', 0, None, 0)
(9, 'artist_hotttnesss', 'REAL', 0, None, 0)
(10, 'year', 'INT', 0, None, 0)
(11, 'track_7digitalid', 'INT', 0, None, 0)
(12, 'shs_perf', 'INT', 0, None, 0)
(13, 'shs_work', 'INT', 0, None, 0)
\First five rows:
('TRMMMYQ128F932D901', 'Silent Night', 'Faster Pussy cat')
('TRMMMKD128F425225D', 'Tanssi vaan', 'Karkkiautomaatti')
('TRMMMRX128F93187D9', 'No One Could Ever', 'Hudson Mohawke')
('TRMMMCH128F425532C', 'Si Vos Querés', 'Yerba Brava')
('TRMMMWA128F426B589', 'Tangle Of Aspens', 'Der Mystic')


<b>Saving the lookup table</b>

In [18]:
TRACK_METADATA_PATH = RAW_DIR / "track_metadata.db"
LOOKUP_PATH = PROCESSED_DIR / "track_metadata_lookup.csv"

with sqlite3.connect(TRACK_METADATA_PATH) as conn:
    conn.execute("""
        CREATE TEMP TABLE target_tracks (
            track_id TEXT NOT NULL,
            split TEXT NOT NULL,
            PRIMARY KEY (track_id, split)
        )
    """)

    conn.executemany(
        "INSERT OR IGNORE INTO target_tracks (track_id, split) VALUES (?, ?)",
        [(str(track_id), "train") for track_id in train_ids]
        + [(str(track_id), "test") for track_id in test_ids]
    )

    metadata_df = pd.read_sql_query("""
        SELECT
            t.track_id,
            t.split,
            s.title,
            s.artist_name,
            s.release,
            s.year
        FROM target_tracks AS t
        LEFT JOIN songs AS s
            ON t.track_id = s.track_id
        ORDER BY t.split, t.track_id
    """, conn)

metadata_df.to_csv(LOOKUP_PATH, index=False)
print("Lookup table saved to:", LOOKUP_PATH)
print("Total track entries:", len(metadata_df))
print("\nFirst five rows:")
display(metadata_df.head())

Lookup table saved to: c:\Users\Mina\Documents\CSU_MSDS\Found of AI\Music-Recommendation-ML-Project\data\processed\track_metadata_lookup.csv
Total track entries: 237662

First five rows:


,track_id,split,title,artist_name,release,year
0,TRAABRX12903CC4816,test,What Child Is This? (Album Version),Plus One,Christmas,2002
1,TRAADFO128F92E1E91,test,In The House,Asher D,Still Kickin',0
2,TRAADQW128F427CE68,test,Prophecy,Xymox,Headclouds,1993
3,TRAADRX12903D0EFE8,test,Sulla buona strada,Riccardo Fogli,Sanremo è sempre Sanremo,1976
4,TRAAEJQ128F92C484E,test,Four Letter Word,Raheem Devaughn,Love Behind The Melody,2008


<b>Checking matching data between the tracks dataset and the tracks metadata db</b>

In [19]:
for split_name in ["train", "test"]:
    subset = metadata_df[metadata_df["split"] == split_name]

    matched = subset["title"].notna() & subset["artist_name"].notna()
    named = (
        subset["title"].fillna("").str.strip().ne("")
        & subset["artist_name"].fillna("").str.strip().ne("")
    )

    total = len(subset)
    matched_count = int(matched.sum())
    named_count = int(named.sum())

    print(f"\n{split_name.upper()} SET")
    print(f"Total tracks: {total:,}")
    print(f"Tracks with metadata fields: {matched_count:,} "
          f"({matched_count / total:.2%})")
    print(f"Tracks with non-empty title and artist: {named_count:,} "
          f"({named_count / total:.2%})")
    print(f"Tracks without complete metadata: {total - named_count:,}")


TRAIN SET
Total tracks: 210,519
Tracks with metadata fields: 210,519 (100.00%)
Tracks with non-empty title and artist: 210,518 (100.00%)
Tracks without complete metadata: 1

TEST SET
Total tracks: 27,143
Tracks with metadata fields: 27,143 (100.00%)
Tracks with non-empty title and artist: 27,142 (100.00%)
Tracks without complete metadata: 1


In [20]:
json_track_ids = {path.stem for path in json_files}
lyric_track_ids = set(train_ids) | set(test_ids)
tagged_track_ids = set()

for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        record = json.load(f)

    if record.get("tags"):
        tagged_track_ids.add(
            str(record.get("track_id", file_path.stem))
        )

shared_tracks = json_track_ids & lyric_track_ids
tracks_with_both = shared_tracks & tagged_track_ids

print(f"JSON catalog: {len(json_track_ids):,}")
print(f"Lyric catalog: {len(lyric_track_ids):,}")
print(f"Tracks in both catalogs: {len(shared_tracks):,}")
print(f"Shared tracks with non-empty tags: {len(tracks_with_both):,}")
print(
    "Percentage of shared tracks with tags:",
    f"{len(tracks_with_both) / len(shared_tracks):.2%}"
)

JSON catalog: 943,334
Lyric catalog: 237,662
Tracks in both catalogs: 231,009
Shared tracks with non-empty tags: 173,730
Percentage of shared tracks with tags: 75.20%


<b>Exploring unique and distinct vocabulary</b>

In [21]:
X_all = vstack([X_train, X_test], format="csr")
unique_terms = np.diff(X_all.indptr)
total_counts = np.asarray(X_all.sum(axis=1)).ravel()

lyric_quality = pd.DataFrame({
    "unique_terms": unique_terms,
    "total_word_counts": total_counts,
})

print("Unique terms per track:")
print(lyric_quality["unique_terms"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50]
))

print("\nTotal word counts per track:")
print(lyric_quality["total_word_counts"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50]
))

print("\nTracks with fewer than 5 unique terms:",
      (lyric_quality["unique_terms"] < 5).sum())

print("Tracks with fewer than 10 unique terms:",
      (lyric_quality["unique_terms"] < 10).sum())

print("Tracks with fewer than 20 total word counts:",
      (lyric_quality["total_word_counts"] < 20).sum())

Unique terms per track:
count    237662.000000
mean         80.136210
std          42.368576
min           1.000000
1%            8.000000
5%           26.000000
10%          38.000000
25%          55.000000
50%          74.000000
max         728.000000
Name: unique_terms, dtype: float64

Total word counts per track:
count    237662.000000
mean        212.939309
std         137.802579
min           1.000000
1%           14.000000
5%           54.000000
10%          80.000000
25%         126.000000
50%         187.000000
max        8578.000000
Name: total_word_counts, dtype: float64

Tracks with fewer than 5 unique terms: 844
Tracks with fewer than 10 unique terms: 2883
Tracks with fewer than 20 total word counts: 3471


In [22]:
X_all = vstack([X_train, X_test], format="csr")
all_ids = list(train_ids) + list(test_ids)
all_splits = (
    ["train"] * len(train_ids)
    + ["test"] * len(test_ids)
)

unique_terms = np.diff(X_all.indptr)
total_counts = np.asarray(X_all.sum(axis=1)).ravel()

metadata_lookup = (
    metadata_df.drop_duplicates("track_id")
    .set_index("track_id")[["title", "artist_name"]]
    .to_dict("index")
)

sparse_samples = []

for i in np.argsort(unique_terms)[:30]:
    row = X_all.getrow(i)
    top_positions = np.argsort(row.data)[::-1][:10]

    top_words = ", ".join(
        f"{vocab[row.indices[j]]} ({row.data[j]})"
        for j in top_positions
    )

    track_id = all_ids[i]
    meta = metadata_lookup.get(track_id, {})

    sparse_samples.append({
        "track_id": track_id,
        "split": all_splits[i],
        "title": meta.get("title"),
        "artist": meta.get("artist_name"),
        "unique_terms": int(unique_terms[i]),
        "total_counts": int(total_counts[i]),
        "most_common_words": top_words,
    })

sparse_review = pd.DataFrame(sparse_samples)
display(sparse_review)

,track_id,split,title,artist,unique_terms,total_counts,most_common_words
0,TRVSBTV128F92E6168,train,La Plata,Jota Quest,1,1,f (1)
1,TRCVJJE12903CD948B,test,Govinda Hare,Krishna Das,1,1,he (1)
2,TRPQHVQ128F92E48BC,train,Sahiller,Rafet El Roman,1,2,bir (2)
3,TRXSJTM128F92E6CBF,test,Aurora,Mike Oldfield,1,3,ei (3)
4,TRPQNPF12903CF31FA,test,Namah Shivaya,Krishna Das,1,2,om (2)
5,TRISWFP128E0782CF0,train,Zeroes (1999 Digital Remaster),David Bowie,1,2,vie (2)
6,TRCLUDQ128F423642C,train,Pulu (2001 Digital Remaster),Ismo Alanko Säätiö,1,1,v (1)
7,TRBUOBU128F148E838,train,Warszawa (Live) (2005 Digital Remaster),David Bowie,1,2,vie (2)
8,TRSVSZS128F92E497C,train,Izanbergaren punta,Berri Txarrak,1,1,mend (1)
9,TRNIKHD12903CB62BE,train,Can't You Sleep,Inchtabokatables,1,150,sheep (150)
